<a href="https://colab.research.google.com/github/satyamkarn100-ctrl/FinDocAI/blob/main/02_LiLt_label_projection_and_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers datasets accelerate seqeval huggingface_hub

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import json
import pandas as pd
from pathlib import Path
from huggingface_hub import snapshot_download

In [ ]:
REPO_ID = "Satyamkarn100/FinDoc-AI"
WORK_DIR = Path("./findoc_work").resolve()
HF_DIR = WORK_DIR / "hf_data"
ANN_DIR = WORK_DIR / "fatura_ann"
WORK_DIR.mkdir(parents=True,exist_ok=True)

snapshot_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    local_dir=HF_DIR,
    allow_patterns=["manifests/*", "ocr/doctr_default/*.jsonl"],
)

manifest = pd.read_csv(HF_DIR / "manifests/manifest.csv")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

In [ ]:
ocr_data = {}
for shard in sorted((HF_DIR / "ocr").rglob("shard_*.jsonl")):
    print("Loading:", shard.name)
    with open(shard, encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)
            ocr_data[rec["filename"]] = rec

print("Total OCR loaded:", len(ocr_data))

Loading: shard_0000.jsonl
Loading: shard_0001.jsonl
Loading: shard_0002.jsonl
Total OCR loaded: 10000


In [ ]:
ORIG_DIR = ANN_DIR / "invoices_dataset_final/Annotations/Original_Format"
if not ORIG_DIR.exists():
    zip_path = WORK_DIR / "FATURA2.zip"
    !wget -c "https://zenodo.org/records/10371464/files/FATURA2.zip?download=1" -O "{zip_path}"
    !unzip -q -o "{zip_path}" "invoices_dataset_final/Annotations/Original_Format/*" -d "{ANN_DIR}"

print("Manifest rows :", len(manifest))
print("OCR loaded    :", len(ocr_data))
print("JSONs loaded  :", len(list(ORIG_DIR.glob("*.json"))))
print("\nSplit counts:\n", manifest["split"].value_counts())

--2026-11-21 17:21:22--  https://zenodo.org/records/10371464/files/FATURA2.zip?download=1
Resolving zenodo.org (zenodo.org)... 137.138.121.78, 188.185.42.113, 188.185.51.60, ...
Connecting to zenodo.org (zenodo.org)|137.138.121.78|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 690717658 (659M) [application/octet-stream]
Saving to: ‘/kaggle/working/findoc_work/FATURA2.zip’

/kaggle/working/fin 100%[===================>] 658.72M  2.88MB/s    in 5m 50s  

2026-11-21 17:27:13 (1.88 MB/s) - ‘/kaggle/working/findoc_work/FATURA2.zip’ saved [690717658/690717658]

Manifest rows : 10000
OCR loaded    : 10000
JSONs loaded  : 10000

Split counts:
 split
train    6400
test     2000
val      1600
Name: count, dtype: int64


In [ ]:
file_path = ORIG_DIR / "Template1_Instance0.json"

with open(file_path,'r') as f:
    ann = json.load(f)

print(ann["TITLE"])

{'bbox': [[328.0, 802.1638], [443.02, 820.1638]], 'text': 'TAX INVOICE'}


In [ ]:
bbox = ann["TITLE"]["bbox"]
x1,y1 = bbox[0]
x2,y2 = bbox[1]

rec = ocr_data['Template1_Instance0.jpg']
w = rec['width']
h = rec['height']


nx1 = min(x1,x2) / w
nx2 = max(x1,x2) / w

ny1 = (h-max(y1,y2)) / h
ny2 = (h-min(y1,y2)) / h

data_box = [nx1,ny1,nx2,ny2]
print(data_box)

[0.5512605042016807, 0.024775505350772845, 0.7445714285714286, 0.04617859690844229]


In [ ]:
gx1, gy1, gx2, gy2 = data_box

for wd in rec["words"]:
    text = wd[0]
    wx1, wy1, wx2, wy2 = wd[2:6]

    ix1 = max(wx1, gx1)
    iy1 = max(wy1, gy1)
    ix2 = min(wx2, gx2)
    iy2 = min(wy2, gy2)

    if ix2 > ix1 and iy2 > iy1:
        inter_area = (ix2 - ix1) * (iy2 - iy1)
        word_area = (wx2 - wx1) * (wy2 - wy1)
        ratio = inter_area / word_area

        if ratio >= 0.5:
            print(text, round(ratio, 2))

TAX 0.97
INVOICE 1.0
